In [0]:
# get the table name from widget (or use default)
dbutils.widgets.text("Database", "sebraepe_dev")
database =  dbutils.widgets.get("Database")

if database not in ["sebraepe_dev", "sebraepe_prod"]:
    raise ValueError("O valor do widget Database deve ser 'sebraepe_dev' ou 'sebraepe_prod' e não pode ser nulo.")

In [0]:
%run /Workspace/sebrae_pe/common/utils/environment

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.types import *

In [0]:
# read table
data = spark.read.option("mergeSchema", "true").parquet(
    "/Volumes/sebraepe_dev/landing/raw/tests/testes_sample.parquet")

In [0]:
display(data.select("Data Liberação", "Data Solicitação", "dthr_entrada"))

Data Liberação,Data Solicitação,dthr_entrada
2024-02-05 13:53:04.255,2024-02-05 08:13:14.101,null
2024-12-12 10:02:26.814,2024-12-05 08:16:56.706,null
2023-11-06 15:06:33.872,2023-08-28 09:05:38.697,null
2025-09-12 09:18:44.048,2025-08-25 09:55:15.462,null
2025-09-30 08:47:33.35,2025-08-12 13:53:47.43,null
2022-09-09 09:44:29.331,2022-08-08 09:45:20.926,null
2022-11-28 13:55:37.533,2022-11-28 08:29:15.863,null
2023-01-09 11:27:09.251,2023-01-04 09:15:40.76,null
2023-04-28 14:00:28.117,2023-04-28 09:54:23.552,null
2023-11-28 11:08:52.219,2023-07-12 14:49:34.788,null


In [0]:
# drop nas in "Valor Resultado" column
data = data.filter(F.col("Valor Resultado").isNotNull())

### Criar variavel plaq count

In [0]:
params = {
    "PLAQUETAS": "PLAQUETAS",
    "NEUTRÓFILOS SEGMENTADOS": "NEUTROFILOS_SEGMENTADOS",
    "EOSINÓFILOS": "EOSINOFILOS",
    "BASÓFILOS": "BASOFILOS",
    "LINFÓCITOS": "LINFOCITOS",
    "LINFÓCITO ATÍPICO": "LINFOCITO_ATIPICO"
}

df = data
df = df.withColumn("param_final", F.lit(None).cast("string"))
for key, val in params.items():
    df = df.withColumn(
        "param_final",
        F.when(F.upper(F.col("Nome do Parâmetro")).contains(key.upper()), val)
        .otherwise(F.col("param_final"))
    )
data = df


In [0]:
data.groupBy("param_final").count().show()

+--------------------+-----+
|         param_final|count|
+--------------------+-----+
|          LINFOCITOS|  174|
|                NULL| 6408|
|   LINFOCITO_ATIPICO|  152|
|         EOSINOFILOS|  143|
|           PLAQUETAS|  175|
|           BASOFILOS|  156|
|NEUTROFILOS_SEGME...|  189|
+--------------------+-----+



In [0]:
# tratamentos dos valores das categorias que nao sao plaqueta

def dividePor100(data, param_column, param_100):
    """ divide por 100 a coluna 'Valor Resultado', 
        apenas nos casos em que a coluna param_100 == param_100
        pressupoe que o vies de ,00 virar algarismo numerico acontece em categorias especificas de 
        parametro de acordo com a estrutura do dataset
    """

    data = data.withColumn(
        "Valor Resultado",
    F.when(
        F.col("Nome do Parâmetro") == param_100,
        F.col("Valor Resultado") / 100
    ).otherwise(F.col("Valor Resultado"))
    
    )

    # verificar
    neutrofilos = data[data["param_final"] == param_column]
    print(neutrofilos.select(["Nome do Parâmetro", "Valor Resultado"]).groupBy(["Nome do Parâmetro"]).mean().show())

    return data

In [0]:
data = dividePor100(data, 
             param_column="NEUTROFILOS_SEGMENTADOS", 
             param_100="[NEUTRÓFILOS SEGMENTADOS %1] * [LEUCÓCITOS1] / 100")

data = dividePor100(data, 
             param_column="BASOFILOS", 
             param_100="[BASÓFILOS %1] * [LEUCÓCITOS1] / 100")

data = dividePor100(data, 
             param_column="EOSINOFILOS", 
             param_100="[EOSINÓFILOS %1] * [LEUCÓCITOS1] / 100")

data = dividePor100(data, 
             param_column="LINFOCITOS", 
             param_100="[LINFÓCITOS %1] * [LEUCÓCITOS1] / 100")

data = dividePor100(data, 
             param_column="LINFOCITOS", 
             param_100='<font size="1">[LINFÓCITOS %1] * [LEUCÓCITOS1] / 100</font>')

data = dividePor100(data, 
             param_column="LINFOCITO_ATIPICO", 
             param_100="[EOSINÓFILOS %1] * [LEUCÓCITOS1] / 100")


data = dividePor100(data, 
             param_column="BASOFILOS", 
             param_100="[BASÓFILOS %1] * [LEUCÓCITOS1] / 100")



+--------------------+--------------------+
|   Nome do Parâmetro|avg(Valor Resultado)|
+--------------------+--------------------+
|<font size="1">[N...|   6070.645502645502|
+--------------------+--------------------+

None
+--------------------+--------------------+
|   Nome do Parâmetro|avg(Valor Resultado)|
+--------------------+--------------------+
|           BASÓFILOS|                 0.0|
|<font size="1">[B...|   40.23841059602649|
+--------------------+--------------------+

None
+--------------------+--------------------+
|   Nome do Parâmetro|avg(Valor Resultado)|
+--------------------+--------------------+
|<font size="1">[E...|    196.006993006993|
+--------------------+--------------------+

None
+--------------------+--------------------+
|   Nome do Parâmetro|avg(Valor Resultado)|
+--------------------+--------------------+
|<font size="1">[L...|            183050.0|
|[LINFÓCITOS %1] *...|              2128.0|
|<font size="1">[L...|  2962.4269005847955|
+-------------

In [0]:
data.count()

7397

In [0]:
# drop nas in "Valor Resultado" column
data = data.filter(F.col("param_final").isNotNull())

In [0]:
def generate_exam_features(data):

    # groupby average value by prontuario + param_final
    w_prompt_param = W.partitionBy("Prontuário", "param_final")

    agg_mean =( 
               (data.withColumn("valor_medio", F.mean("Valor Resultado").over(w_prompt_param))
                .select(
                    F.col("Prontuário").alias("prontuario"),
                    "param_final",
                    "valor_medio"
                    )
        ).groupBy("prontuario")
        .pivot("param_final")
        .agg(F.first("valor_medio"))
    )

    
    # renomeamos as colunas para adicionar os sufixos
    for col in agg_mean.columns[1:]:
        agg_mean = agg_mean.withColumnRenamed(col, f"{col.lower()}_valor_medio")

    # groupby max date value by prontuario + param_final
    data = data.withColumn("data_lib_ts", F.to_timestamp(F.col("Data Liberação")))  # garante timestamp
    w_orderdate = W.partitionBy("Prontuário", "param_final").orderBy(F.col("data_lib_ts").desc())

    agg_max_date = (
                (data.withColumn("rn", F.row_number().over(w_orderdate))
                .filter(F.col("rn") == 1)
                .select(
                    F.col("Prontuário").alias("prontuario"),
                    "param_final",
                    F.col("Valor Resultado").alias("valor_max_date")
                )
        ).groupBy("prontuario")
        .pivot("param_final")
        .agg(F.first("valor_max_date"))
    )

    for col in agg_max_date.columns[1:]:
        agg_max_date = agg_max_date.withColumnRenamed(col, f"{col.lower()}_valor_max_date")


    # join the two datasets
    features = agg_mean.join(agg_max_date, on="prontuario", how="inner")

    return features

🔹 Ambiente definido: dev
🔹 Catálogo ativo: sebraepe_dev


### Aplicar criação de variáveis por janelas temporais

In [0]:

# construir meses de referencia a partir das datas minima e maxima
bounds = data.agg(
    F.date_trunc("month", F.min("Data Liberação")).alias("min_m"),
    F.date_trunc("month", F.max("Data Liberação")).alias("max_m"),
)

ref_dates_df = bounds.select(
    F.expr("sequence(min_m, max_m, interval 1 month) as ref_dates")
).select(F.explode("ref_dates").alias("ref_date"))

# lista de ref dates distintos
ref_dates = [r.ref_date for r in ref_dates_df.collect()]

In [0]:
# define dataframe para incorporar dados ao cursor
features = spark.createDataFrame([], schema=StructType())

# para cada data de referencia, filtrar os dados anteriores a ela, para calculo das variaveis
for ref_ts in ref_dates:

    data_filtered = data.filter(
        (F.col("Data Liberação") < F.lit(ref_ts)) &
        (F.col("Data Liberação") >= F.add_months(F.lit(ref_ts), -12))
    )
    
    features_ref_date = generate_exam_features(data_filtered)

    features_ref_date = features_ref_date.withColumn("date_ref", F.lit(ref_ts).cast("timestamp"))

    features = features.unionByName(features_ref_date, allowMissingColumns=True)

    print(ref_ts)

2022-08-01 00:00:00
2022-08-08 00:00:00
2022-08-15 00:00:00
2022-08-22 00:00:00
2022-08-29 00:00:00
2022-09-05 00:00:00
2022-09-12 00:00:00
2022-09-19 00:00:00
2022-09-26 00:00:00
2022-10-03 00:00:00
2022-10-10 00:00:00
2022-10-17 00:00:00
2022-10-24 00:00:00
2022-10-31 00:00:00
2022-11-07 00:00:00
2022-11-14 00:00:00
2022-11-21 00:00:00
2022-11-28 00:00:00
2022-12-05 00:00:00
2022-12-12 00:00:00
2022-12-19 00:00:00
2022-12-26 00:00:00
2023-01-02 00:00:00
2023-01-09 00:00:00
2023-01-16 00:00:00
2023-01-23 00:00:00
2023-01-30 00:00:00
2023-02-06 00:00:00
2023-02-13 00:00:00
2023-02-20 00:00:00
2023-02-27 00:00:00
2023-03-06 00:00:00
2023-03-13 00:00:00
2023-03-20 00:00:00
2023-03-27 00:00:00
2023-04-03 00:00:00
2023-04-10 00:00:00
2023-04-17 00:00:00
2023-04-24 00:00:00
2023-05-01 00:00:00
2023-05-08 00:00:00
2023-05-15 00:00:00
2023-05-22 00:00:00
2023-05-29 00:00:00
2023-06-05 00:00:00
2023-06-12 00:00:00
2023-06-19 00:00:00
2023-06-26 00:00:00
2023-07-03 00:00:00
2023-07-10 00:00:00
